# Software profesional en Acústica 2025-26 (M2i)

*This notebook contains a modification of the notebook [FEM_Helmholtz_equation_Robin](https://github.com/spatialaudio/computational_acoustics/blob/master/FEM_Helmholtz_equation_Robin.ipynb), created by Sascha Spors, Frank Schultz, Computational Acoustics Examples, 2018. The text/images are licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/). The FEniCS implementation has been replaced with NGSolve. The code is released under the [MIT license](https://opensource.org/licenses/MIT).*

This Jupyter Notebook has been designed to be run in [Google Colab](https://colab.research.google.com/). With this purpose the first cell install [NGSolve](https://ngsolve.org/) related packages in a clean machine (if they have not been previously installed). Typically, this installation takes less than 2 minutes.

In [ ]:
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-complex.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

# Numerical Solution of the coupling two fluid domains by using an impedance boundary condition associated to a veil or local reacting rigid plate

This notebook illustrates the numerical solution of the wave equation for an harmonic excitation coupling a fluid domain with porous materials using the [Finite Element Method](https://en.wikipedia.org/wiki/Finite_element_method) (FEM). The method aims at an approximate solution by subdividing the area of interest into smaller parts with simpler geometry, linking these parts together and applying methods from the calculus of variations to solve the problem numerically. The FEM is a well established method for the numerical approximation of the solution of partial differential equations (PDEs). The solutions of PDEs are often known analytically only for rather simple geometries. FEM based simulations allow to gain insights into other more complex cases such this coupled fluid-porous problem.

## Problem Statement

The inhomogeneous vector-valued Helmholtz equation governs the displacement field in the fluid domain $\Omega_{\mathrm{F}}$ and it is given as

\begin{equation}
\rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\nabla\text{div}\, U(\mathbf{x}, \omega) + \omega^2\rho_{\mathrm{F}} U(\mathbf{x}, \omega) = - Q(\mathbf{x}, \omega) \qquad\text{in }\Omega_{\mathrm{F}}.
\end{equation}

The fluid domain is split in two parts separated by a impedance boundary condition located on the coupling interface $\Gamma_{\mathrm{I}}$. These coupling conditions between the porous and fluid domains is expressed in terms of the kinetic and kinematic boundary conditions,the continuity of the normal displacements on the coupling boundary and a jump condition on the pressure values, this is,

\begin{equation}
P(\mathbf{x}^{+}, \omega)-P(\mathbf{x}^{-}, \omega) = -i\omega Z_{\mathrm{S}}(\omega)\mathbf{U}(\mathbf{x}, \omega)\cdot\mathbf{n},\qquad \mathbf{x}\in\Gamma_{\mathrm{I}}.
\end{equation}

where $Z_{\mathrm{S}}(\omega)$ is the complex-valued surface impedance associated with the porous veil or the local-reacting rigid plate. In the case of the porous veil $Z_{\mathrm{S}}(\omega) = \alpha+i\beta/\omega$, where $\alpha$ and $\beta$ are constants. For the local-reacting rigid plate, $Z_{\mathrm{S}}(\omega) = (-\omega^2 m -i\omega r +s)/(-i\omega)$, where $m$, $r$, and $s$ are respectively the surface mass density, the damping and the stiffness associated with the rigid plate. The boundary conditions are completed assuming rigid boundaries on the rest of the boundary of the computational domain.

## Variational Formulation

Starting from the variational formulation of the Helmholtz equation (before application of Green's first theorem)

\begin{align*}
\int_{\Omega_{\mathrm{F}}} \rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\text{div}\, U(\mathbf{x}, \omega)\text{div}\, V(\mathbf{x}, \omega) \mathrm{d}x  - \omega^2\rho_{\mathrm{F}}\int_{\Omega_{\mathrm{F}}} U(\mathbf{x}, \omega)\cdot V(\mathbf{x}, \omega) \mathrm{d}x - i\omega Z_{\mathrm{S}}(\omega)\int_{\Gamma_{\mathrm{I}}} U(\mathbf{x}, \omega)\cdot\mathbf{n}\ V(\mathbf{x}, \omega)\cdot\mathbf{n} \mathrm{d}x
= -\int_{\Omega_{\mathrm{F}}} Q(\mathbf{x}, \omega) V(\mathbf{x}, \omega) \mathrm{d}x
\end{align*}


It is common to express this integral equation in terms of the bilinear $a(P, V)$ and linear $L(V)$ forms 

\begin{equation}
a(U, V) = \int_{\Omega_{\mathrm{F}}} \rho_{\mathrm{F}}c_{\mathrm{F}}^{2}\text{div}\, U(\mathbf{x}, \omega)\text{div}\, V(\mathbf{x}, \omega) \mathrm{d}x  - \omega^2\rho_{\mathrm{F}}\int_{\Omega_{\mathrm{F}}} U(\mathbf{x}, \omega)\cdot V(\mathbf{x}, \omega) \mathrm{d}x - i\omega Z_{\mathrm{S}}(\omega)\int_{\Gamma_{\mathrm{I}}} U(\mathbf{x}, \omega)\cdot\mathbf{n}\  V(\mathbf{x}, \omega)\cdot\mathbf{n} \mathrm{d}x
\end{equation}

\begin{equation}
L(V) = -\int_{\Omega_{\mathrm{F}}} Q(\mathbf{x}, \omega) V(\mathbf{x}, \omega) \mathrm{d}x,
\end{equation}

where the variational problem to solve is stated as: Find $U$ such that

\begin{equation}
a(U, V) = L(V)\qquad \forall V.
\end{equation}

## Numerical Solution

The numerical solution of the variational problem is based on [NGSolve](https://ngsolve.org/), an open-source framework for numerical solution of PDEs.
Its high-level Python interface is used in the following to define the problem and compute the solution.
The implementation is based on the variational formulation derived above.
It is common in the FEM to denote the solution of the problem by $\boldsymbol{U}(\boldsymbol{x})$ and the test function by $\boldsymbol{V}(\boldsymbol{x})$.
The definition of the problem in NGSolve is very close to the mathematical formulation of the problem.

For the subsequent examples the solution of inhomogeneous wave equation for a radial vector-valued source centered at a point $\boldsymbol{x}_s$: $F(\boldsymbol{x}) = \exp(-\mu ||\boldsymbol{x}-\boldsymbol{x}_s||)\boldsymbol{r}_{s}(\boldsymbol{x})$ where $\boldsymbol{r}_{s}(\boldsymbol{x})=(\boldsymbol{x}-\boldsymbol{x}_s)/||\boldsymbol{x}-\boldsymbol{x}_s||$

In [ ]:
import ngsolve as ng
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
import numpy as np
import matplotlib.pyplot as plt


def fluid_impedance(mesh, frequency, xs, rhoF, bulkF, alpha=0., beta=0.):
    """
    Solve the vector Helmholtz equation for a displacement field in a fluid
    domain split by an impedance interface at x=3.25.

    Parameters
    ----------
    mesh      : NGSolve Mesh
    frequency : float, excitation frequency in Hz
    xs        : (x, y) tuple/list with the point-source location
    alpha     : float, real part of surface impedance
    beta      : float, coefficient for imaginary part (Zs = alpha + 1j*beta/omega)

    Returns
    -------
    sol : ngsolve.GridFunction (complex)
    """

    # Physical parameters
    omega_val = 2 * np.pi * frequency
    Zs    = alpha + 1j * beta / omega_val  # complex surface impedance

    # First-order Raviart-Thomas space (HDiv in NGSolve)
    V = HDiv(mesh, order=2, RT=True, complex=True)

    u, v = V.TnT()   # trial and test functions

    # Bilinear form:  a(u,v) = (div u, div v) - k^2 (u, v)
    a = BilinearForm(V)
    a += bulkF * div(u) * div(v) * dx - omega_val**2 * rhoF * InnerProduct(u, v) * dx - Zs * InnerProduct(u.Trace(),v.Trace()) * ds("interface")
    a.Assemble()

    # ── linear form  L(v) = radial vector-valued Gaussian pulse at x=xs  ─────────────────────────
    xp, yp = xs
    radial_vector = CF(((x - xp)/sqrt((x - xp) ** 2 + (y - yp) ** 2) * exp(-5* sqrt((x - xp) ** 2 + (y - yp) ** 2)), 
                         (y - yp)/sqrt((x - xp) ** 2 + (y - yp) ** 2)* exp(-5* sqrt((x - xp) ** 2 + (y - yp) ** 2))))
    f = LinearForm(V)
    f += InnerProduct(radial_vector, v) * dx
    f.Assemble()

    # Build solution GridFunction
    sol = GridFunction(V)

    # ── solve ───────────────────────────────────────────────────────────────

    # Sparse direct solver
    solver = Preconditioner(a,"direct")

    # Solve the acoustic problem at t=0
    solvers.BVP(bf=a, lf=f, gf=sol, pre=solver, print=False)

    return sol

### Sound Field transmitted through a layer of porous material

Two 2D-dimensional rooms are connected by a layer of porous material. The porous layer decrease the amplitude of the pressure field and the sound power level is computed on both rooms to check the efficiency absorbing the incident pressure generated by a point source in the left room

In [ ]:
# Build geometry: two rectangular rooms connected through a narrow channel at x=3.25
# Left (input) room:  [0,3] x [0,4]  +  channel [3, 3.25] x [1.5, 2.5]
# Right (output) room: [3.5,6] x [0,4]  +  channel [3.25, 3.5] x [1.5, 2.5]

# Build geometry using Netgen
domain_left = MoveTo(0., 0.).Rectangle(3., 4.).Face()
domain_right = MoveTo(3.5, 0.).Rectangle(3., 4.).Face()
channel_left = MoveTo(3., 1.5).Rectangle(0.25, 1.).Face()
channel_right = MoveTo(3.25, 1.5).Rectangle(0.25, 1.).Face()

domain_input= domain_left + channel_left
domain_output= domain_right + channel_right

# Domain and boundary tags for the PML domain
domain_input.faces.name = "input"
domain_input.faces.col = (0, 1, 0)  # Green 
domain_input.mat("input")
domain_input.edges.name = "exterior"
domain_input.edges.col = (1, 1, 0) 
domain_input.edges.Max(X).name = "interface"
domain_input.edges.Max(X).col = (1, 0, 0) # red

# Domain and boundary tags for the fluid domain
domain_output.faces.name = "output"
domain_output.faces.col = (0, 0, 1)  # Green 
domain_output.mat("output")
domain_output.edges.name = "exterior"
domain_output.edges.col = (1, 1, 0) 
domain_output.edges.Min(X).name = "interface"
domain_output.edges.Min(X).col = (1, 0, 0) # red

# Generate geomerey
geo = OCCGeometry(Glue([domain_input, domain_output]), dim=2)

# Generate mesh
ngmesh = geo.GenerateMesh(maxh=0.1)
mesh = Mesh(ngmesh)
mesh.Curve(3)  # Curve order 3 for better approximation of the circle

Draw(mesh, height="3vh")
print("Number of vertices:", mesh.nv)

The two-dimensional sound field in a rectangular room (rectangular plate) with homogeneous Robin boundary conditions is computed for the frequency $f=120$ Hz and source position $x_s = (1.2,3.2)$ m.

In [ ]:
f = 120.0        # frequency [Hz]
xs = (1.2, 3.2)  # source position
rhoF  = 1.21           # fluid mass density [kg/m^3]
bulkF = rhoF * 343**2  # bulk modulus [Pa]

# Compute solution without surface impedance (alpha=0, beta=0)
gfu = fluid_impedance(mesh, f, xs, rhoF, bulkF, alpha=0., beta=0.)

# plot the modulus of the pressure field
Draw(Norm(gfu), mesh, height="3vh")

# plot the the pressure field animated with the phase values
Draw(gfu[0]**2+gfu[1]**2, mesh, "solution", animate_complex=True, height="3vh")

Compute the sound velocity level (SVL) in the output and input room

In [ ]:
v_ref = 5e-8  # reference velocity [m/s]
omega_val = 2.0 * np.pi * f

# Integrate over each subdomain using domain labels
SVL_input  = 20 * np.log10(omega_val * Integrate(Norm(gfu), mesh, definedon=mesh.Materials("input"))  / v_ref)
SVL_output = 20 * np.log10(omega_val * Integrate(Norm(gfu), mesh, definedon=mesh.Materials("output")) / v_ref)

print('SVL in the input room without surface impedance:', SVL_input,  'dB')
print('SVL in the output room without surface impedance:', SVL_output, 'dB')

In [ ]:
f = 120.0        # frequency [Hz]
xs = (1.2, 3.2)  # source position
rhoF  = 1.21           # fluid mass density [kg/m^3]
bulkF = rhoF * 343**2  # bulk modulus [Pa]

# Compute solution with surface impedance (alpha=1e3, beta=5e2)
gfu = fluid_impedance(mesh, f, xs,  rhoF, bulkF, alpha=1e3, beta=5e2)

# plot the modulus of the pressure field
Draw(Norm(gfu), mesh, height="3vh")

# plot the the pressure field animated with the phase values
Draw(gfu[0]**2+gfu[1]**2, mesh, "solution", animate_complex=True, height="3vh")

Compute the SVL values associated with the input and output room, and the insertion loss (IL) values 

In [ ]:
v_ref = 5e-8  # reference velocity [m/s]
omega_val = 2.0 * np.pi * f

SVL_input_impedance  = 10 * np.log10(omega_val * Integrate(Norm(gfu), mesh, definedon=mesh.Materials("input"))  / v_ref)
SVL_output_impedance = 10 * np.log10(omega_val * Integrate(Norm(gfu), mesh, definedon=mesh.Materials("output")) / v_ref)

print('SVL in the input room with surface impedance:',  SVL_input_impedance,  'dB')
print('SVL in the output room with surface impedance:', SVL_output_impedance, 'dB')

# Insertion loss for the particle velocity
IL = SVL_output - SVL_output_impedance
print('Insertion loss (IL):', IL, 'dB')

### Exercise 
Replace the porous veil with a rigid plate. Please, analyze the effect of varying the mass density, the stiffness and the damping coefficient by computing the frequency response of SVL in the range between $5$ Hz and $500$ Hz.

In [ ]:
## YOUR CODE HERE

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT).